In [ ]:
"""
Project 13 — Move 1 Step 4 / Phase 2: Compositional Expression Model (dual-stream cis+trans)

Manager handoff (2026-07-16), Engineer build. Gated on Move 1 Step 3's POC SUCCESS
(scbloodnl_move1_poc_report.json: bifurcation_readout=SUCCESS, trans beats cis
p=8.3e-6 across 21/24 folds) -- per the project's own rule, the full architecture
only gets built after the lightweight POC clears. It has (2026-07-15i backfill,
STATE.md).

WHAT THIS IS: a real dual-stream neural net (1D-CNN cis-sequence encoder +
MLP trans-TF-state encoder -> fusion head) predicting condition-specific gene
expression delta (target_delta_expression), trained/evaluated with the SAME
leave-one-condition-out design as Step 3's POC (notebooks/kaggle_1m_scbloodnl_move1_poc.py)
so Phase 2's Pearson r can be compared fold-by-fold against Step 3's cis/trans
baselines -- an apples-to-apples upgrade check, not just a new number in isolation.

TWO CORRECTIONS vs the original Manager handoff spec (flagged in STATE.md, not
silently changed):
  1. Condition names in the handoff spec (Mono_Candida_3h, Mono_Flu_24h, ...)
     don't exist in 1M-scBloodNL. Real codes: 3hCA/24hCA/3hMTB/24hMTB/3hPA/24hPA
     (Candida / M. tuberculosis / Pseudomonas aeruginosa) x 4 cell types
     (B, monocyte, CD8T, naive_CD4T). No flu condition in this dataset.
  2. Uses leave-one-condition-out across all 4 groups x 6 conditions (24 folds),
     not a single fixed train/test split, to match Step 3's design.

REAL INPUTS (not the mock generator from the handoff spec):
  - x_trans: that (cell type, condition)'s delta values for the 71
     FALLBACK_TF_LIST genes (same list Step 3 used), read straight from
     results/kaggle_1m_scbloodnl_move1_poc results/scbloodnl_move1_delta_{group}.parquet.
  - x_cis: real hg38 2000bp promoter windows (TSS-1500/+500, strand-aware),
     one-hot encoded (A=[1,0,0,0], C=[0,1,0,0], G=[0,0,1,0], T=[0,0,0,1], N=[0,0,0,0]),
     fetched via UCSC's api.genome.ucsc.edu REST endpoint (network-reachable,
     confirmed from the dev sandbox; Kaggle has internet when enabled),
     using gene_coords.csv + data/processed/gene_symbol_to_ensg.csv for the
     symbol -> ENSG -> coordinate mapping. Cached locally to a single .npz so a
     full-genome fetch only ever happens once.
  - y: target_delta_expression, straight from the same delta parquet (the
     ground-truth column Step 3 predicted).

DEPENDENCIES (documented per repo convention -- pin/install at top of any
fresh Kaggle session):
    pip install torch pandas numpy scipy pyarrow requests
Tested against: torch>=2.0, pandas>=2.0, numpy>=1.24, scipy>=1.10.

VERSIONING: this is v1. Do not overwrite -- next iteration is
kaggle_phase2_compositional_model_v2.py per repo file-versioning convention.

OUTPUTS (mirrors Step 3's output shape for direct comparison):
  - phase2_compositional_folds.csv        (per-fold r_trans-style Pearson r, one row per group x held-out condition)
  - phase2_compositional_report.json     (pooled summary + bifurcation-style readout)
  - figures/phase2_loss_curves_<group>_<held_out>.png (one per fold, high-res)
  - dna_promoter_cache.npz                (gene -> one-hot promoter array, reusable across folds/groups)

NOT YET RUN ON KAGGLE -- logic-validated in-sandbox on a small gene subset only
(see the __main__ SMOKE_TEST block). Full run needs Eric to execute on Kaggle:
promoter fetch for the full ~18-22k gene panel + CNN training across 24 folds
is well past this sandbox's 45s-per-call / no-persistent-process ceiling.
"""

import json
import os
import time
import urllib.parse
import urllib.request
import concurrent.futures
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from scipy.stats import pearsonr
from torch.utils.data import DataLoader, Dataset

# =====================================================================
# 0. PATHS & CONSTANTS
# =====================================================================

# Adjusted to match the exact flat directory structure from your scan
REPO_ROOT = Path("/kaggle/input/datasets/ericdu847/kaggle-phase2-compositional-model-v1")

# Inputs are located directly within the mapped REPO_ROOT
DELTA_DIR = REPO_ROOT
GENE_COORDS_PATH = REPO_ROOT / "gene_coords.csv"
SYM2ENSG_PATH = REPO_ROOT / "gene_symbol_to_ensg.csv"

# Outputs MUST point to /kaggle/working to be writable
OUTPUT_DIR = Path("/kaggle/working")
RESULTS_DIR = OUTPUT_DIR / "results"
FIGURES_DIR = OUTPUT_DIR / "figures"
DNA_CACHE_PATH = OUTPUT_DIR / "dna_promoter_cache.npz"

GROUPS = ["B", "monocyte", "CD8T", "naive_CD4T"]  # same 4 cell types as Step 3
SEQ_LEN = 2000
UPSTREAM = 1500   # bp upstream of TSS
DOWNSTREAM = 500  # bp downstream of TSS  (1500 + 500 = 2000bp window)

# Same 71-TF fallback list as Step 3 (notebooks/kaggle_1m_scbloodnl_move1_poc.py)
# so x_trans is directly comparable to what the POC used as its trans fingerprint.
FALLBACK_TF_LIST = [
    "STAT1", "STAT2", "STAT3", "STAT4", "STAT5A", "STAT5B", "STAT6", "IRF1", "IRF3", "IRF4",
    "IRF7", "IRF8", "NFKB1", "NFKB2", "RELA", "RELB", "REL", "JUN", "JUNB", "JUND", "FOS",
    "FOSB", "FOSL1", "FOSL2", "ATF3", "ATF4", "NR4A1", "NR4A2", "NR4A3", "EGR1", "EGR2",
    "EGR3", "MYC", "MYB", "GATA1", "GATA2", "GATA3", "TBX21", "RORC", "FOXP3", "SPI1",
    "CEBPA", "CEBPB", "CEBPD", "IKZF1", "IKZF2", "IKZF3", "BCL6", "PRDM1", "BATF", "BATF3",
    "EOMES", "RUNX1", "RUNX3", "TCF7", "LEF1", "ID2", "ID3", "KLF2", "KLF4", "KLF6",
    "SOCS1", "SOCS3", "PPARG", "XBP1", "ETS1", "ETS2", "ELF1", "ELK1", "MAF", "MAFB",
]

BASE_ONEHOT = {
    "A": [1, 0, 0, 0], "C": [0, 1, 0, 0], "G": [0, 0, 1, 0], "T": [0, 0, 0, 1],
}


# =====================================================================
# 1. REAL CIS INPUT: hg38 PROMOTER FETCH + ONE-HOT
# =====================================================================

def fetch_promoter_dna(chrom, tss, strand, upstream=UPSTREAM, downstream=DOWNSTREAM,
                       retries=3, timeout=10):
    """Fetch a strand-aware [-upstream, +downstream] window around TSS from
    UCSC's REST API (hg38). Returns the raw sequence string (mixed-case is
    fine -- soft-masked repeat lowercase is normalized to uppercase before
    one-hot encoding, we don't treat repeat-masking specially here)."""
    if strand == "+":
        start, end = tss - upstream, tss + downstream
    else:
        start, end = tss - downstream, tss + upstream

    start = max(1, start)
    url = (f"https://api.genome.ucsc.edu/getData/sequence?genome=hg38;"
           f"chrom={chrom};start={start};end={end}")
    for attempt in range(retries):
        try:
            with urllib.request.urlopen(url, timeout=timeout) as resp:
                payload = json.loads(resp.read().decode())
            seq = payload.get("dna", "")
            if not seq:
                return None
            seq = seq.upper()
            if strand == "-":
                comp = str.maketrans("ACGT", "TGCA")
                seq = seq.translate(comp)[::-1]
            return seq
        except Exception as e:
            if attempt == retries - 1:
                print(f"[fetch_promoter_dna] FAILED {chrom}:{start}-{end} after {retries} tries: {e}")
                return None
            time.sleep(1.5 * (attempt + 1))


def one_hot_encode(seq, length=SEQ_LEN):
    """Sequence string -> (length, 4) one-hot array. Pads/truncates to a fixed
    length (matches the model's fixed Conv1d input). Non-ACGT bases (N, etc.)
    encode to the zero vector, matching the handoff spec's default-padding rule."""
    arr = np.zeros((length, 4), dtype=np.float32)
    seq = seq[:length]
    for i, base in enumerate(seq):
        if base in BASE_ONEHOT:
            arr[i] = BASE_ONEHOT[base]
    return arr


def load_gene_coords():
    coords = pd.read_csv(GENE_COORDS_PATH)  # gene(ENSG), chrom, strand, gene_start, gene_end
    sym2ensg = pd.read_csv(SYM2ENSG_PATH)   # symbol, gene_id(ENSG)
    merged = sym2ensg.merge(coords, left_on="gene_id", right_on="gene", how="inner")
    merged["tss"] = np.where(merged["strand"] == "+", merged["gene_start"], merged["gene_end"])
    return merged.set_index("symbol")[["chrom", "strand", "tss"]]


def fetch_single_gene_worker(gene, coords_lookup):
    """Worker task designed to handle the coordinate lookup and api call for one gene."""
    if gene not in coords_lookup.index:
        return gene, np.zeros((SEQ_LEN, 4), dtype=np.float32)
    
    row = coords_lookup.loc[gene]
    if isinstance(row, pd.DataFrame):
        row = row.iloc[0]
        
    seq = fetch_promoter_dna(row["chrom"], int(row["tss"]), row["strand"])
    if seq is None:
        return gene, np.zeros((SEQ_LEN, 4), dtype=np.float32)
    
    return gene, one_hot_encode(seq)


def build_dna_dict(gene_symbols, cache_path=DNA_CACHE_PATH, max_workers=16):
    """Real dna_dict: gene_symbol -> (2000, 4) one-hot promoter array.
    Uses multi-threaded concurrency (ThreadpoolExecutor) to fast-fetch UCSC APIs.
    Caches to a single .npz keyed by gene symbol so repeated runs never re-fetch."""
    cache_path = Path(cache_path)
    cache_path.parent.mkdir(parents=True, exist_ok=True)
    cache = {}
    if cache_path.exists():
        with np.load(cache_path) as npz:
            cache = {k: npz[k] for k in npz.files}
        print(f"[build_dna_dict] loaded {len(cache)} cached promoters from {cache_path}")

    coords_lookup = load_gene_coords()
    missing = [g for g in gene_symbols if g not in cache]
    print(f"[build_dna_dict] {len(missing)} genes need fetching (of {len(gene_symbols)} requested)")

    if missing:
        n_completed = 0
        # Fetch missing genes in parallel to speed up extraction from ~15 mins to ~1 min
        with concurrent.futures.ThreadPoolExecutor(max_workers=max_workers) as executor:
            future_to_gene = {
                executor.submit(fetch_single_gene_worker, gene, coords_lookup): gene 
                for gene in missing
            }
            for future in concurrent.futures.as_completed(future_to_gene):
                gene, one_hot_arr = future.result()
                cache[gene] = one_hot_arr
                n_completed += 1
                if n_completed % 50 == 0 or n_completed == len(missing):
                    print(f"[build_dna_dict] {n_completed}/{len(missing)} fetched parallelly...")

        np.savez_compressed(cache_path, **cache)
        print(f"[build_dna_dict] cache updated: {len(cache)} total genes -> {cache_path}")

    return {g: cache[g] for g in gene_symbols}


# =====================================================================
# 2. REAL TRANS INPUT + TARGETS: from Step 3's delta parquets
# =====================================================================

def load_group_delta(group):
    """Same source Step 3 used: results/.../scbloodnl_move1_delta_{group}.parquet.
    Index = gene symbol, columns = the 6 stim conditions, values = delta(gene) =
    mean(condition) - mean(UT) pseudobulk expression."""
    path = DELTA_DIR / f"scbloodnl_move1_delta_{group}.parquet"
    return pd.read_parquet(path)


def build_samples_and_tf_dict(delta_df, tf_genes):
    """samples_df: long-format (gene_id, condition_id, target_delta_expression).
    tf_dict: condition_id -> (71,) array of that condition's delta for the TF genes
    (the real trans "weather report" -- same quantity Step 3 used as its trans
    fingerprint, just fed to a learned MLP here instead of a similarity weight)."""
    # Clean NaN values from base delta dataframe before setting up matrices
    delta_df = delta_df.fillna(0.0).replace([np.inf, -np.inf], 0.0)
    
    tf_present = [g for g in tf_genes if g in delta_df.index]
    tf_block = delta_df.loc[tf_present]  # (n_tf, n_conditions)

    tf_dict = {}
    for cond in delta_df.columns:
        vec = np.zeros(len(tf_genes), dtype=np.float32)
        vals = tf_block[cond].values
        vec[: len(vals)] = vals
        tf_dict[cond] = vec

    samples = delta_df.reset_index().melt(
        id_vars=delta_df.index.name or "index",
        var_name="condition_id", value_name="target_delta_expression"
    ).rename(columns={delta_df.index.name or "index": "gene_id"})
    
    # Ensure no NaN target values pass into loss calculation
    samples = samples.dropna(subset=["target_delta_expression"]).reset_index(drop=True)
    samples = samples[np.isfinite(samples["target_delta_expression"])].reset_index(drop=True)
    return samples, tf_dict, len(tf_genes)


# =====================================================================
# 3. DATASET (real data, same shapes as the handoff spec)
# =====================================================================

class SingleCellCompositionalDataset(Dataset):
    def __init__(self, samples_df, dna_dict, tf_dict, num_tfs):
        self.samples = samples_df.reset_index(drop=True)
        self.dna_dict = dna_dict
        self.tf_dict = tf_dict
        self.num_tfs = num_tfs

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        row = self.samples.iloc[idx]
        gene_id = row["gene_id"]
        condition_id = row["condition_id"]
        target_expression = row["target_delta_expression"]

        x_cis = torch.tensor(
            self.dna_dict.get(gene_id, np.zeros((SEQ_LEN, 4), dtype=np.float32)),
            dtype=torch.float32,
        )
        x_trans = torch.tensor(
            self.tf_dict.get(condition_id, np.zeros(self.num_tfs, dtype=np.float32)),
            dtype=torch.float32,
        )
        y = torch.tensor([target_expression], dtype=torch.float32)
        return x_cis, x_trans, y


# =====================================================================
# 4. MODEL (verbatim from the Manager handoff spec with epsilon stability)
# =====================================================================

class CisSequenceEncoder(nn.Module):
    def __init__(self, sequence_length=SEQ_LEN, embedding_dim=128):
        super().__init__()
        self.conv_block = nn.Sequential(
            nn.Conv1d(4, 64, kernel_size=15, stride=1, padding=7),
            nn.BatchNorm1d(64, eps=1e-5), nn.ReLU(), nn.MaxPool1d(4),
            nn.Conv1d(64, 128, kernel_size=9, stride=1, padding=4),
            nn.BatchNorm1d(128, eps=1e-5), nn.ReLU(), nn.MaxPool1d(4),
            nn.Conv1d(128, 256, kernel_size=5, stride=1, padding=2),
            nn.BatchNorm1d(256, eps=1e-5), nn.ReLU(), nn.AdaptiveAvgPool1d(1),
        )
        self.fc = nn.Sequential(nn.Linear(256, embedding_dim), nn.ReLU(), nn.Dropout(0.2))

    def forward(self, x):
        if x.shape[1] != 4:
            x = x.transpose(1, 2)
        x = self.conv_block(x).squeeze(-1)
        return self.fc(x)


class TransStateEncoder(nn.Module):
    def __init__(self, num_tfs=71, embedding_dim=64):
        super().__init__()
        self.mlp = nn.Sequential(
            nn.Linear(num_tfs, 128), nn.BatchNorm1d(128, eps=1e-5), nn.ReLU(), nn.Dropout(0.2),
            nn.Linear(128, embedding_dim), nn.ReLU(), nn.Dropout(0.1),
        )

    def forward(self, x):
        return self.mlp(x)


class CompositionalExpressionModel(nn.Module):
    def __init__(self, num_tfs=71, cis_emb_dim=128, trans_emb_dim=64):
        super().__init__()
        self.cis_encoder = CisSequenceEncoder(embedding_dim=cis_emb_dim)
        self.trans_encoder = TransStateEncoder(num_tfs=num_tfs, embedding_dim=trans_emb_dim)
        fusion_dim = cis_emb_dim + trans_emb_dim
        self.fusion_head = nn.Sequential(
            nn.Linear(fusion_dim, 128), nn.ReLU(), nn.Dropout(0.3),
            nn.Linear(128, 64), nn.ReLU(), nn.Dropout(0.2),
            nn.Linear(64, 1),
        )

    def forward(self, x_cis, x_trans):
        h_cis = self.cis_encoder(x_cis)
        h_trans = self.trans_encoder(x_trans)
        h_fusion = torch.cat([h_cis, h_trans], dim=1)
        return self.fusion_head(h_fusion)


# =====================================================================
# 5. TRAIN/EVAL ONE LEAVE-ONE-CONDITION-OUT FOLD
# =====================================================================

def run_fold(group, held_out_condition, dna_dict, epochs=15, batch_size=32,
             lr=5e-4, weight_decay=0.01, device=None, save_loss_plot=True):
    device = device or ("cuda" if torch.cuda.is_available() else "cpu")

    delta_df = load_group_delta(group)
    samples, tf_dict, num_tfs = build_samples_and_tf_dict(delta_df, FALLBACK_TF_LIST)

    train_samples = samples[samples["condition_id"] != held_out_condition]
    test_samples = samples[samples["condition_id"] == held_out_condition]

    train_ds = SingleCellCompositionalDataset(train_samples, dna_dict, tf_dict, num_tfs)
    test_ds = SingleCellCompositionalDataset(test_samples, dna_dict, tf_dict, num_tfs)
    train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True, drop_last=True)
    test_loader = DataLoader(test_ds, batch_size=batch_size, shuffle=False)

    model = CompositionalExpressionModel(num_tfs=num_tfs).to(device)
    criterion = nn.HuberLoss()
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=weight_decay)

    train_losses = []
    for epoch in range(epochs):
        model.train()
        epoch_loss, n_batches = 0.0, 0
        for x_cis, x_trans, y in train_loader:
            x_cis, x_trans, y = x_cis.to(device), x_trans.to(device), y.to(device)
            optimizer.zero_grad()
            pred = model(x_cis, x_trans)
            loss = criterion(pred, y)
            
            # Protect optimizer steps from infinite/NaN components
            if not torch.isnan(loss) and not torch.isinf(loss):
                loss.backward()
                # Apply gradient clipping to completely secure weights from vanishing/exploding
                torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
                optimizer.step()
                epoch_loss += loss.item()
                n_batches += 1
                
        train_losses.append(epoch_loss / max(n_batches, 1))

    model.eval()
    preds, targets = [], []
    with torch.no_grad():
        for x_cis, x_trans, y in test_loader:
            x_cis, x_trans = x_cis.to(device), x_trans.to(device)
            pred = model(x_cis, x_trans).cpu().numpy().flatten()
            preds.extend(pred.tolist())
            targets.extend(y.numpy().flatten().tolist())

    # Safely calculate correlation metric by pruning potential static predictions
    if len(preds) > 1 and np.std(preds) > 1e-9 and np.std(targets) > 1e-9:
        r, p = pearsonr(preds, targets)
    else:
        r, p = 0.0, 1.0

    if save_loss_plot:
        _save_loss_curve(group, held_out_condition, train_losses)

    return {
        "group": group,
        "held_out_condition": held_out_condition,
        "n_train": len(train_samples),
        "n_test": len(test_samples),
        "n_tf": int((delta_df.reindex(FALLBACK_TF_LIST).notna().any(axis=1)).sum()),
        "r_phase2": r,
        "p_phase2": p,
        "final_train_loss": train_losses[-1] if train_losses else float("nan"),
    }


def _save_loss_curve(group, held_out_condition, losses):
    import matplotlib
    matplotlib.use("Agg")
    import matplotlib.pyplot as plt

    FIGURES_DIR.mkdir(parents=True, exist_ok=True)
    fig, ax = plt.subplots(figsize=(6, 4), dpi=200)
    ax.plot(range(1, len(losses) + 1), losses, marker="o", linewidth=1.5)
    ax.set_xlabel("Epoch")
    ax.set_ylabel("Train Huber loss")
    ax.set_title(f"Phase 2 dual-stream: {group} held out {held_out_condition}")
    ax.grid(alpha=0.3)
    fig.tight_layout()
    out_path = FIGURES_DIR / f"phase2_loss_curve_{group}_{held_out_condition}.png"
    fig.savefig(out_path)
    plt.close(fig)


# =====================================================================
# 6. MAIN: run all folds, write report (mirrors Step 3's report shape)
# =====================================================================

def main(groups=GROUPS, epochs=15, smoke_test_n_genes=None):
    RESULTS_DIR.mkdir(parents=True, exist_ok=True)

    # Universe of genes across all groups (delta parquets share the same gene index
    # per Step 3's design -- confirmed: all 4 groups indexed by the same ~22,150 genes).
    all_genes = load_group_delta(groups[0]).index.tolist()
    if smoke_test_n_genes:
        all_genes = all_genes[:smoke_test_n_genes]
        print(f"[main] SMOKE TEST MODE: restricting to first {smoke_test_n_genes} genes")

    dna_dict = build_dna_dict(all_genes)

    fold_results = []
    for group in groups:
        delta_df = load_group_delta(group)
        conditions = list(delta_df.columns)
        for held_out in conditions:
            print(f"[main] running fold: group={group}, held_out={held_out}")
            res = run_fold(group, held_out, dna_dict, epochs=epochs)
            print(f"[main]   -> r_phase2={res['r_phase2']:.4f} (n_test={res['n_test']})")
            fold_results.append(res)

    folds_df = pd.DataFrame(fold_results)
    folds_df.to_csv(RESULTS_DIR / "phase2_compositional_folds.csv", index=False)

    mean_r = float(np.nanmean(folds_df["r_phase2"]))
    n_over_035 = int((folds_df["r_phase2"] > 0.35).sum())
    report = {
        "n_folds": len(folds_df),
        "mean_r_phase2": mean_r,
        "n_folds_r_over_0.35": n_over_035,
        "target_threshold": 0.35,
        "bifurcation_readout": (
            "SUCCESS -- Phase 2 mean held-out Pearson r exceeds the 0.35 target."
            if mean_r > 0.35 else
            "NOT CLEARED -- Phase 2 mean held-out Pearson r below the 0.35 target."
        ),
        "per_fold": fold_results,
    }
    with open(RESULTS_DIR / "phase2_compositional_report.json", "w") as f:
        json.dump(report, f, indent=2, default=float)

    print(f"[main] DONE. mean_r_phase2={mean_r:.4f}, {n_over_035}/{len(folds_df)} folds > 0.35")
    print(f"[main] outputs -> {RESULTS_DIR}")
    return report


if __name__ == "__main__":
    # SMOKE TEST: small gene subset, few epochs, ONE fold only -- logic validation
    # before the full Kaggle run (same discipline Step 3's POC followed). Full run
    # (all genes, all 24 folds, epochs=15) is the Kaggle job.
    SMOKE_TEST = False
    if SMOKE_TEST:
        print("=== SMOKE TEST: 200 genes, 1 group, 1 fold, 2 epochs ===")
        genes = load_group_delta("B").index.tolist()[:200]
        dna_dict = build_dna_dict(genes)
        result = run_fold("B", "24hCA", dna_dict, epochs=2, save_loss_plot=True)
        print(json.dumps(result, indent=2, default=float))
    else:
        main()